# YOLOv8 Training Notebook (Ultralytics + Roboflow)

Notebook ini adalah template untuk:
- download dataset dari Roboflow
- training YOLOv8 dengan Ultralytics
- validasi dan inference hasil model

**Catatan:** ganti placeholder API key / workspace / project / version sesuai dataset kamu.

## 1) Install dependency
Jalankan cell ini di Google Colab atau Jupyter.

In [ ]:
!nvidia-smi

Mon Apr 20 09:03:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q ultralytics roboflow
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import os
from ultralytics import YOLO
print('Ultralytics installed successfully')

True
Tesla T4
Ultralytics installed successfully


## 2) Download dataset dari Roboflow
Paste snippet Roboflow kamu di bawah ini. Jika dataset sudah diekspor dalam format YOLOv8, pastikan `download('yolov8')`.

In [ ]:
# ==============================
# ROBOFLOW DATASET DOWNLOAD
# ==============================
# Replace these placeholders with your Roboflow info.
# Example:
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')
# version = project.version(YOUR_VERSION)
# dataset = version.download('yolov8')
# data_yaml = os.path.join(dataset.location, 'data.yaml')

from roboflow import Roboflow

ROBOFLOW_API_KEY = 'YOUR_API_KEY'
WORKSPACE_NAME = 'YOUR_WORKSPACE_NAME'
PROJECT_NAME = 'YOUR_PROJECT_NAME'
VERSION_NUMBER = 14

#bagian ini pake snip code dari roboflownya
rf = Roboflow(api_key="YOUR_API_KEU")
project = rf.workspace("YOUR_WORKSPACE_NAME").project("YOUR_PROJECT_NAME")
version = project.version(YOUR_VERSION_NUMBER)
dataset = version.download("yolov8") 


data_yaml = os.path.join(dataset.location, 'data.yaml')
print('Dataset location:', dataset.location)
print('data.yaml:', data_yaml)

loading Roboflow workspace...
loading Roboflow project...
Dataset location: /content/rock-paper-scissors-14
data.yaml: /content/rock-paper-scissors-14/data.yaml


## 3) Cek struktur dataset
Cell ini membantu memastikan path dataset sudah benar.

In [ ]:
import os

print('Files in dataset folder:')
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in files[:10]:
        print(f'{subindent}{f}')

Files in dataset folder:
rock-paper-scissors-14/
  README.dataset.txt
  data.yaml
  README.roboflow.txt
  train/
    labels.cache
    images/
      zoom_tests_RockPaperScissors-mohamed_mp4-173_jpg.rf.78dbeca1760925c798622c5a661dce49.jpg
      bistro_40_15_altavista_jpg.rf.123fd02ab7344427b093b47901e6a6ec.jpg
      IMG_7079_MOV-148_jpg.rf.503864f921c3017b6e057eca716159d8.jpg
      JWitt-RCP_mp4-31_jpg.rf.6ed21353a78728e69e4c06dc4d98f364.jpg
      egohands-public-1621214707543_png_jpg.rf.809f4092a7873a0d425df93bbcba24dd.jpg
      egohands-public-1625254250331_png_jpg.rf.0b8622617e6348d94f0e5b8cd8379434.jpg
      0010_png.rf.b0b4cc31d98f0006e41b6bb59d22ce24.jpg
      IMG_7043_MOV-211_jpg.rf.005573778518a1a96bbcd8564e49e7e2.jpg
      egohands-public-1625679150457_png_jpg.rf.d4ebc94f550f94846e8d731357dd76b9.jpg
      IMG_5636_MOV-6_jpg.rf.de0438b838b1a37254441693874b20b2.jpg
    labels/
      Screen-Shot-2021-05-12-at-3-28-39-PM_png_jpg.rf.935ded218238f81f0a294c9d4635929c.txt
      egohands

## 4) Load model YOLOv8
Pakai model kecil dulu untuk training awal.

In [ ]:
# Choose one:
# yolov8n.pt = nano, paling ringan
# yolov8s.pt = small, masih relatif ringan
# yolov8m.pt = medium

model = YOLO('yolov8s.pt')
print('Model loaded:', model.model_name if hasattr(model, 'model_name') else 'YOLOv8n')

Model loaded: yolov8s.pt


## 5) Training
Atur epoch, image size, batch size, dan device sesuai resource.

In [ ]:
# Ubah parameter 'name' untuk mengganti nama folder hasil
# Ubah parameter 'project' untuk mengganti folder utama penyimpanannya

results = model.train(
    data=data_yaml,
    epochs=15, 
    imgsz=640,
    batch=64, # untuk tesla T4 batch maksimal adalah 64
    patience=20,
    device=0, # menggunakan GPU pertama yang terdeteksi pada hardware
    project='runs/train',      # Folder utama
    name='yolov8_roboflow'   # Nama spesifik eksperimen ini
)

print(results)

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rock-paper-scissors-14/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_roboflow-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 6) Validation
Evaluasi model setelah training selesai.

In [ ]:
metrics = model.val(data=data_yaml)
print(metrics)

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1058.2±270.1 MB/s, size: 29.5 KB)
val: Scanning /content/rock-paper-scissors-14/valid/labels.cache... 576 images, 238 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 576/576 142.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.6it/s 9.9s
                   all        576        400      0.924      0.935      0.947      0.745
                 Paper        132        139      0.866       0.95      0.948      0.742
                  Rock        121        141      0.942      0.957       0.94      0.724
              Scissors        116        120      0.964      0.899      0.953       0.77
Speed: 2.5ms preprocess, 9.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detec

## Tips praktis
- Mulai dari `yolov8n.pt` dulu.
- Pastikan dataset Roboflow sudah dalam format YOLOv8.
- jika GPU Colab terbatas, kecilkan batch size.
- Setelah tra|ining, cek `runs/train/.../weights/best.pt`.


## Praktikum

1. cari dataset dari roboflow (boleh buat, boleh ngambil)
2. ambil sniped codenya dan lakukan training disini
3. cari hasil best.pt pada /content/runs/detect/runs/train/yolov8_roboflow/weights/best.pt
4. cari hasil graph, matrix, kurva, dll, yg berupa hasil dari training. (terletak di /content/runs/detect/runs/train/yolov8_roboflow)


# Tugas
1. cari hasil best.pt
2. lakukan inferensi dengan model baru hasil training. gunakan python dan openCV. (ada bounding box dan label classnya terbaca)
3. Lakukan pendalaman mengenai hasil2 graph, curve, dan matrix. cari tahu cara bacanya yaaa!
4. pada bagian 5) training, lakukan pendalaman mengenai parameternya (epoch, batch, dll) apa pengaruhnya pada training. 

# Tips
1. kalau cari/buat dataset yg kecil aja, jangan banyak2. tp bebas sih
2. Kecilkan epoch, lama soalnya :(
3. pake google colab GPU T4 yaa



refrensi:

https://docs.ultralytics.com/modes/train/

https://youtu.be/a3SBRtILjPI?si=GcsQ___h7Xo9Vp6S

https://youtu.be/r0RspiLG260?si=f2UzNq5CTQ0dIaLF

